In [2]:
"""
Expanded Algospeak & Nigerian/African Coded Discourse Detection Pipeline
Topic: Machine Learning Detection of Global Algospeak and Culturally Embedded
       Nigerian/African Digital Coded Expressions

Author-use note:
- This script updates the earlier algospeak pipeline by adding Nigerian Pidgin,
  African digital slang, political/economic coded terms, religious discourse,
  sex/body euphemisms, and youth internet expressions.
- Not every expression here is "algospeak" in the strict moderation-evasion sense.
  Some are community-coded, euphemistic, metaphorical, or culturally embedded forms.
- The labels therefore separate global algospeak from Nigerian/African coded discourse.
"""

import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA, LatentDirichletAllocation
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DPI = 600
FIGDIR = "figures_expanded_algospeak"
os.makedirs(FIGDIR, exist_ok=True)

N_PER_CLASS = 450

LABEL_ORDER = [
    "global_algospeak",
    "sexual_body",
    "nigerian_pidgin",
    "religious_spiritual",
    "politics_economy",
    "youth_culture",
    "ambiguous",
    "standard",
]

PALETTE = {
    "global_algospeak": "#D62828",
    "sexual_body": "#9D4EDD",
    "nigerian_pidgin": "#F77F00",
    "religious_spiritual": "#2A9D8F",
    "politics_economy": "#1D3557",
    "youth_culture": "#457B9D",
    "ambiguous": "#E9C46A",
    "standard": "#6C757D",
}

# =============================================================================
# EXPANDED ALGORITHMIC / CODED LANGUAGE LEXICON
# =============================================================================

GLOBAL_ALGOSPEAK = [
    "seggs", "s*xual", "$uicide", "sewer slide", "unalived", "corn", "pr0n",
    "le$bian", "SA", "grape", "k!ll", "unalive", "passed", "candy", "broccoli",
    "co-vid", "v@ccine", "pew-pew", "spicy content", "spicy accountant", "yt",
    "pdf file", "sad brain", "shadow-banned", "spicy opinion", "adult content",
    "cens0red", "b@n", "clock app", "picture app", "self-delete",
]

SEXUAL_BODY = [
    "kpeku", "kpekus", "gbola", "knack", "do the thing", "ashawo", "runs girl",
    "sugar daddy", "odogwu", "press phone", "belle don enter", "carry belle",
    "monthly visitor", "stand well", "collect lips", "enter body",
    "untouched land", "toast", "scope", "sponsor", "cruise partner", "hot cake",
    "yansh", "oranges", "backside", "enter room", "tear rubber",
    "complete package", "target", "boo boo", "main plug", "cut rope",
    "silent follower", "private part", "package", "sleep over", "body no be wood",
]

NIGERIAN_PIDGIN = [
    "kpai", "carry go", "aza", "big man", "boo", "yahoo", "yahoo boy",
    "carry belle", "don go", "rest", "iron", "agbóro", "five-o", "kirikiri",
    "igbo", "colos", "money ritual", "bad market", "suffer-head", "soft life",
    "fine girl no pimple", "babe", "side chick", "breakfast", "served breakfast",
    "wahala", "shine your eye", "commot", "packaging", "aproko", "mumu",
    "body dey hot", "belle dey cry", "chop life", "spray money", "trekker",
    "dry", "settle", "share the cake", "oga at the top", "cabal", "long leg",
    "plug", "plug man", "gbege", "gbas gbos", "scatter ground", "dust leg",
    "cap", "no cap", "cast", "don cast", "ghost", "format", "die laugh",
    "kpele", "my g", "gee", "my guy", "oga", "chairman", "baller",
    "calm down small", "i feel you", "na true", "no lies", "madt", "choke",
    "burst brain", "crack ribs", "e shock me", "weak me", "no cast me",
    "i don fold", "take am easy", "no gree", "flex", "dey ball", "spray am",
    "free me", "cool temper", "use your head", "no sabi", "sharp guy",
    "sabi person", "too fine", "machine", "who know who", "dodge bullet",
    "gbese", "enter gbese", "e don click", "e no enter",
]

RELIGIOUS_SPIRITUAL = [
    "village people", "brother", "sister", "baba god", "man of god",
    "divine breakthrough", "enemies of progress", "fire prayer", "spiritual battle",
    "baba", "arrow", "open door", "mountain", "old serpent", "evil load",
    "god when", "grace locate you", "monitoring spirit", "agents",
    "strange movement", "breakthrough enter", "testimony land", "favor locate",
    "open heaven", "god run am", "scatter heaven", "heaven answer",
    "audio miracle", "soft life ministry",
]

POLITICS_ECONOMY = [
    "money na water", "t-pain", "naija", "fuel wahala", "obodo hard",
    "shege pro max", "subsidy don comot", "e cost die", "hunger dey wire person",
    "account dey cry", "aza dry", "money don enter", "find daily bread",
    "trenches", "streets", "street no get joy", "suffer suffer",
    "national cake eater", "chop national cake", "audio promise", "audio money",
    "audio work", "steady billing", "billing", "send transport", "show love",
    "sponsor ministry", "pressure wan kill person", "drag person",
    "collect online beating", "missing funds", "write result", "chop chairman",
    "appreciate somebody", "government lodge", "invite", "carry am",
    "hard times", "market don rise", "one chance", "update work", "hustler",
    "know person",
]

YOUTH_CULTURE = [
    "premium tears", "breakfast and lunch", "baba for the boys", "landlord",
    "spec", "hot spec", "find person", "shoot shot", "bounce person",
    "disgrace am for junction", "play ball", "clear road", "carry chair go house",
    "red card", "yellow card", "wahala pro max", "agent of chaos",
    "instagram life", "oppress us", "pepper dem", "alert enter", "alert land",
    "aza drop", "hustle mode", "don blow", "celeb material",
    "trend for wrong reason", "drag season", "day one", "snake under grass",
    "battery low", "speed of light", "street certified", "don see shege",
    "slay", "periodt", "bestie", "no cap", "fr fr", "lowkey", "highkey",
]

STANDARD_SEEDS = [
    "content moderation policies are too strict",
    "platform censorship debate is ongoing",
    "free speech online has limits",
    "social media algorithms prioritize engagement",
    "misinformation spreads faster than facts",
    "digital literacy is important today",
    "online harassment affects mental health",
    "community guidelines need reform",
    "big technology companies shape public discourse",
    "user data privacy is frequently violated",
    "recommendation systems create echo chambers",
    "viral content shapes public opinion",
    "language policing on social platforms is increasing",
    "banned words lists keep growing",
    "creator economy is unfairly punished",
    "demonetization affects marginalized voices",
    "shadow banning affects small accounts",
    "engagement metrics drive content visibility",
    "automated moderation makes errors",
    "human reviewers are often overworked",
    "artificial intelligence cannot always detect context",
    "false positives harm creators",
    "marginalized groups face more censorship",
    "sexual health education should be available online",
    "political activism is shaped by algorithms",
    "platform governance requires transparency",
    "speech classification systems need accountability",
    "language technologies often favor dominant languages",
]

AMBIGUOUS_SEEDS = [
    "doing the thing tonight with friends",
    "you know what I mean right",
    "the community really needs this resource",
    "let us not talk about that here",
    "send me that information privately",
    "my account keeps getting flagged randomly",
    "I cannot say the word but you know",
    "the algorithm dislikes this kind of content",
    "speaking in code because of bans",
    "my last video was removed again",
    "cannot post this openly anymore",
    "creative language for sensitive topics",
    "euphemism culture is growing online",
    "self censorship is now common",
    "platform specific vocabulary is emerging",
    "linguistic adaptation is needed online",
    "new words are replacing old banned ones",
    "community members find workarounds",
    "coded language protects the community",
    "implicit meaning appears in social posts",
    "reading between the lines is necessary",
    "plausible deniability appears in captions",
    "strategic ambiguity is common online",
    "hashtag replacement strategy works",
    "letter substitution is becoming common",
    "emoji can replace sensitive words",
    "numbers replace letters in posts",
    "asterisks appear in the middle of words",
    "misspellings are understood by the group",
    "deliberate typos help avoid detection",
    "phonetic spelling replaces sensitive content",
    "homophone replacement appears in posts",
    "spaces appear inside sensitive words",
    "community created vocabulary changes quickly",
]

LEXICON_BY_CLASS = {
    "global_algospeak": GLOBAL_ALGOSPEAK,
    "sexual_body": SEXUAL_BODY,
    "nigerian_pidgin": NIGERIAN_PIDGIN,
    "religious_spiritual": RELIGIOUS_SPIRITUAL,
    "politics_economy": POLITICS_ECONOMY,
    "youth_culture": YOUTH_CULTURE,
}

ALL_CODED_TERMS = sorted(set(
    GLOBAL_ALGOSPEAK
    + SEXUAL_BODY
    + NIGERIAN_PIDGIN
    + RELIGIOUS_SPIRITUAL
    + POLITICS_ECONOMY
    + YOUTH_CULTURE
), key=len, reverse=True)

# =============================================================================
# TEXT GENERATION HELPERS
# =============================================================================

def normalize_term(term: str) -> str:
    return re.sub(r"\s+", " ", term.lower().strip())


def make_seed_sentence(term: str, label: str) -> str:
    """Convert a coded term into a natural-looking social media sentence."""
    templates = {
        "global_algospeak": [
            "people now say {term} to avoid moderation",
            "this platform flags normal words so users write {term}",
            "the word {term} appears in coded online speech",
            "creators use {term} when discussing sensitive topics",
        ],
        "sexual_body": [
            "people use {term} in informal relationship talk",
            "the expression {term} appears in coded body discourse",
            "youths sometimes say {term} in private online jokes",
            "the term {term} is used as a euphemistic sexual expression",
        ],
        "nigerian_pidgin": [
            "for naija online talk people say {term}",
            "this pidgin expression {term} carries community meaning",
            "many users write {term} in everyday digital discourse",
            "the coded phrase {term} appears in Nigerian social media talk",
        ],
        "religious_spiritual": [
            "religious users describe the situation as {term}",
            "the phrase {term} appears in spiritual online discourse",
            "church communities often understand {term} as coded meaning",
            "people use {term} in faith based digital communication",
        ],
        "politics_economy": [
            "people use {term} when discussing politics and hardship",
            "the phrase {term} appears in Nigerian economic discourse",
            "online users say {term} to comment on public life",
            "the coded term {term} circulates in political discussion",
        ],
        "youth_culture": [
            "young people use {term} in social media interaction",
            "the phrase {term} belongs to youth digital culture",
            "online users say {term} as a community expression",
            "the expression {term} carries informal social meaning",
        ],
    }
    return np.random.choice(templates[label]).format(term=term)


def build_class_seeds(label: str, terms: list[str]) -> list[str]:
    seeds = []
    for term in terms:
        term = normalize_term(term)
        seeds.append(make_seed_sentence(term, label))
        seeds.append(f"{term} is a recognizable expression in online discourse")
    return seeds


def augment_texts(seeds: list[str], n_total: int, label: str) -> list[str]:
    """Generate a synthetic but transparent corpus from seed examples."""
    modifiers = [
        "honestly", "literally", "actually", "tbh", "ngl", "imo", "fr fr",
        "no cap", "lowkey", "highkey", "bestie", "pls", "omg", "lmao", "bruh",
    ]
    platforms = [
        "on TikTok", "on X", "on Instagram", "on Reddit", "on YouTube",
        "on WhatsApp", "on Facebook",
    ]
    discourse_markers = {
        "global_algospeak": ["algorithm", "content moderation", "flagged", "shadow ban"],
        "sexual_body": ["relationship", "body talk", "private gist", "coded talk"],
        "nigerian_pidgin": ["naija", "pidgin", "street talk", "online gist"],
        "religious_spiritual": ["church", "prayer", "testimony", "faith talk"],
        "politics_economy": ["economy", "politics", "hardship", "public discourse"],
        "youth_culture": ["youth slang", "social media", "online culture", "trend"],
        "standard": ["platform policy", "digital speech", "moderation", "public debate"],
        "ambiguous": ["coded meaning", "indirect talk", "private context", "unclear meaning"],
    }

    texts = list(seeds)
    while len(texts) < n_total:
        seed = np.random.choice(seeds)
        op = np.random.choice(["prepend", "append", "platform", "marker"])
        if op == "prepend":
            txt = f"{np.random.choice(modifiers)} {seed}"
        elif op == "append":
            txt = f"{seed} {np.random.choice(modifiers)}"
        elif op == "platform":
            txt = f"{seed} {np.random.choice(platforms)}"
        else:
            txt = f"{seed} {np.random.choice(discourse_markers[label])}"
        texts.append(txt)
    return texts[:n_total]


def add_ambiguity_noise(texts: list[str], labels: list[str], noise_level: float = 0.10):
    """Create realistic surface ambiguity without changing labels."""
    texts = list(texts)
    labels = list(labels)
    n_noise = int(len(texts) * noise_level)
    idx = np.random.choice(len(texts), n_noise, replace=False)

    for i in idx:
        words = texts[i].split()
        if len(words) > 6:
            drop = np.random.randint(1, 3)
            texts[i] = " ".join(words[drop:])
        if np.random.rand() < 0.35:
            texts[i] = texts[i].replace("coded", "kind of").replace("expression", "thing")
    return texts, labels


# =============================================================================
# BUILD CORPUS
# =============================================================================

def build_corpus(n_per_class: int = N_PER_CLASS) -> pd.DataFrame:
    rows = []

    for label, terms in LEXICON_BY_CLASS.items():
        seeds = build_class_seeds(label, terms)
        texts = augment_texts(seeds, n_per_class, label)
        rows.extend({"text": t, "label": label} for t in texts)

    standard_texts = augment_texts(STANDARD_SEEDS, n_per_class, "standard")
    rows.extend({"text": t, "label": "standard"} for t in standard_texts)

    ambiguous_texts = augment_texts(AMBIGUOUS_SEEDS, n_per_class, "ambiguous")
    rows.extend({"text": t, "label": "ambiguous"} for t in ambiguous_texts)

    df = pd.DataFrame(rows)
    noisy_texts, noisy_labels = add_ambiguity_noise(df["text"].tolist(), df["label"].tolist(), noise_level=0.12)
    df = pd.DataFrame({"text": noisy_texts, "label": noisy_labels})
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return df


df = build_corpus(N_PER_CLASS)
print(f"Corpus size: {len(df)} texts")
print(df["label"].value_counts())


# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

def count_phrase_matches(text: str, terms: list[str]) -> int:
    t = normalize_term(text)
    return sum(1 for term in terms if normalize_term(term) in t)


def extract_features(texts: list[str]) -> pd.DataFrame:
    substitution_patterns = ["0", "1", "@", "$", "!", "*", "#", "|", "\\", "/", "3", "@"]
    rows = []

    for text in texts:
        t = text.lower()
        words = t.split()

        row = {
            "char_count": len(text),
            "word_count": len(words),
            "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0,
            "unique_word_ratio": len(set(words)) / max(len(words), 1),
            "substitution_count": sum(t.count(p) for p in substitution_patterns),
            "special_char_density": sum(1 for c in text if not c.isalnum() and c != " ") / max(len(text), 1),
            "caps_ratio": sum(1 for c in text if c.isupper()) / max(len(text), 1),
            "punct_density": sum(1 for c in text if c in "!?.,;:") / max(len(words), 1),
            "emoji_count": sum(1 for c in text if ord(c) > 127),
            "has_leetspeak": int(any(p in text for p in ["0", "1", "@", "$", "3", "!", "*"])),
            "has_pidgin_marker": int(count_phrase_matches(t, NIGERIAN_PIDGIN) > 0),
            "has_global_algospeak": int(count_phrase_matches(t, GLOBAL_ALGOSPEAK) > 0),
            "has_sexual_marker": int(count_phrase_matches(t, SEXUAL_BODY) > 0),
            "has_religious_marker": int(count_phrase_matches(t, RELIGIOUS_SPIRITUAL) > 0),
            "has_political_marker": int(count_phrase_matches(t, POLITICS_ECONOMY) > 0),
            "has_youth_marker": int(count_phrase_matches(t, YOUTH_CULTURE) > 0),
            "global_algospeak_count": count_phrase_matches(t, GLOBAL_ALGOSPEAK),
            "sexual_body_count": count_phrase_matches(t, SEXUAL_BODY),
            "nigerian_pidgin_count": count_phrase_matches(t, NIGERIAN_PIDGIN),
            "religious_spiritual_count": count_phrase_matches(t, RELIGIOUS_SPIRITUAL),
            "politics_economy_count": count_phrase_matches(t, POLITICS_ECONOMY),
            "youth_culture_count": count_phrase_matches(t, YOUTH_CULTURE),
            "all_coded_marker_count": count_phrase_matches(t, ALL_CODED_TERMS),
        }
        rows.append(row)

    return pd.DataFrame(rows)


feat_df = extract_features(df["text"].tolist())
feat_df["label"] = df["label"].values
print("Feature matrix:", feat_df.shape)


# =============================================================================
# TRAIN/TEST SPLIT
# =============================================================================

X_text = df["text"].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)


# =============================================================================
# MODELS
# =============================================================================

models = {
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=8000, lowercase=True)),
        ("clf", MultinomialNB()),
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=8000, lowercase=True)),
        ("clf", LogisticRegression(max_iter=1500, C=1.0, random_state=RANDOM_STATE)),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=8000, lowercase=True)),
        ("clf", LinearSVC(max_iter=3000, C=1.0, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=6000, lowercase=True)),
        ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "Gradient Boosting": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=6000, lowercase=True)),
        ("clf", GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}

print("\nTraining models...")
for name, model in models.items():
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring="f1_weighted",
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    results[name] = {
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "test_accuracy": report["accuracy"],
        "weighted_precision": report["weighted avg"]["precision"],
        "weighted_recall": report["weighted avg"]["recall"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "y_pred": y_pred,
        "report": report,
        "cv_scores": cv_scores,
    }

    print(
        f"{name:22s} | CV F1={cv_scores.mean():.3f}±{cv_scores.std():.3f} "
        f"| Test Acc={report['accuracy']:.3f} | Weighted F1={report['weighted avg']['f1-score']:.3f}"
    )

best_model_name = max(results, key=lambda k: results[k]["weighted_f1"])
best_model = models[best_model_name]
print(f"\nBest model: {best_model_name}")


# =============================================================================
# SAVE TABLES
# =============================================================================

summary_df = pd.DataFrame({
    name: {
        "CV F1 Mean": r["cv_mean"],
        "CV F1 SD": r["cv_std"],
        "Test Accuracy": r["test_accuracy"],
        "Weighted Precision": r["weighted_precision"],
        "Weighted Recall": r["weighted_recall"],
        "Weighted F1": r["weighted_f1"],
    }
    for name, r in results.items()
}).T

summary_df.to_csv(os.path.join(FIGDIR, "model_performance_summary.csv"))

lexicon_rows = []
for label, terms in LEXICON_BY_CLASS.items():
    for term in sorted(set(terms)):
        lexicon_rows.append({"category": label, "coded_expression": term})
pd.DataFrame(lexicon_rows).to_csv(os.path.join(FIGDIR, "expanded_algospeak_lexicon.csv"), index=False)

df.to_csv(os.path.join(FIGDIR, "synthetic_expanded_algospeak_corpus.csv"), index=False)
feat_df.to_csv(os.path.join(FIGDIR, "sociolinguistic_feature_matrix.csv"), index=False)


# =============================================================================
# FIGURE 1: CORPUS OVERVIEW
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor("white")

counts = df["label"].value_counts().reindex(LABEL_ORDER)
colors = [PALETTE[k] for k in LABEL_ORDER]

axes[0].bar(range(len(counts)), counts.values, color=colors, edgecolor="white")
axes[0].set_xticks(range(len(counts)))
axes[0].set_xticklabels([x.replace("_", "\n") for x in counts.index], rotation=0, fontsize=8)
axes[0].set_title("A. Class distribution", fontweight="bold")
axes[0].set_ylabel("Number of posts")

lengths = df.assign(char_count=df["text"].str.len())
for label in LABEL_ORDER:
    subset = lengths[lengths["label"] == label]["char_count"]
    axes[1].hist(subset, bins=25, alpha=0.45, label=label.replace("_", " "), color=PALETTE[label])
axes[1].set_title("B. Post length distribution", fontweight="bold")
axes[1].set_xlabel("Character count")
axes[1].set_ylabel("Frequency")
axes[1].legend(fontsize=7)

marker_cols = [
    "global_algospeak_count",
    "sexual_body_count",
    "nigerian_pidgin_count",
    "religious_spiritual_count",
    "politics_economy_count",
    "youth_culture_count",
]
marker_means = feat_df.groupby("label")[marker_cols].mean().reindex(LABEL_ORDER)
axes[2].imshow(marker_means.values, aspect="auto")
axes[2].set_xticks(range(len(marker_cols)))
axes[2].set_xticklabels([m.replace("_count", "").replace("_", "\n") for m in marker_cols], fontsize=7)
axes[2].set_yticks(range(len(LABEL_ORDER)))
axes[2].set_yticklabels([l.replace("_", " ") for l in LABEL_ORDER], fontsize=8)
axes[2].set_title("C. Mean coded-marker counts", fontweight="bold")

#plt.suptitle("Figure 1. Expanded corpus overview", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig1_expanded_corpus_overview.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FIGURE 2: MODEL PERFORMANCE
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("white")

plot_df = summary_df[["Test Accuracy", "Weighted F1", "CV F1 Mean"]]
x = np.arange(len(plot_df.index))
width = 0.25

for i, col in enumerate(plot_df.columns):
    ax.bar(x + i * width, plot_df[col].values, width=width, label=col)

ax.set_xticks(x + width)
ax.set_xticklabels(plot_df.index, rotation=25, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
#ax.set_title("Figure 2. Classifier performance comparison", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig2_model_performance.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FIGURE 3: CONFUSION MATRIX FOR BEST MODEL
# =============================================================================

best_pred = results[best_model_name]["y_pred"]
cm = confusion_matrix(y_test, best_pred, labels=LABEL_ORDER)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(LABEL_ORDER)))
ax.set_yticks(range(len(LABEL_ORDER)))
ax.set_xticklabels([x.replace("_", "\n") for x in LABEL_ORDER], fontsize=8)
ax.set_yticklabels([x.replace("_", " ") for x in LABEL_ORDER], fontsize=8)

for i in range(len(LABEL_ORDER)):
    for j in range(len(LABEL_ORDER)):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}\n(n={cm[i, j]})", ha="center", va="center", fontsize=6)

ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
#ax.set_title(f"Figure 3. Normalized confusion matrix ({best_model_name})", fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig3_confusion_matrix.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FIGURE 4: DISCRIMINATIVE TF-IDF FEATURES
# =============================================================================

lr_model = models["Logistic Regression"]
lr_model.fit(X_train, y_train)
vectorizer = lr_model.named_steps["tfidf"]
clf = lr_model.named_steps["clf"]
feature_names = vectorizer.get_feature_names_out()

classes = list(clf.classes_)
n_cols = 4
n_rows = int(np.ceil(len(classes) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 12))
axes = axes.flatten()

for ax, class_name in zip(axes, classes):
    class_idx = classes.index(class_name)
    coefs = clf.coef_[class_idx]
    top_idx = np.argsort(coefs)[-12:][::-1]
    top_feats = [feature_names[i] for i in top_idx]
    top_vals = [coefs[i] for i in top_idx]

    y_pos = np.arange(len(top_feats))
    ax.barh(y_pos, top_vals, color=PALETTE.get(class_name, "#333333"))
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_feats, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(class_name.replace("_", " ").title(), fontsize=10, fontweight="bold")
    ax.set_xlabel("Coefficient", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

for ax in axes[len(classes):]:
    ax.axis("off")

#plt.suptitle("Figure 4. Top discriminative TF-IDF features by class", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig4_tfidf_features.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FIGURE 5: SOCIOLINGUISTIC FEATURE DISTRIBUTIONS
# =============================================================================

plot_features = [
    ("global_algospeak_count", "Global algospeak"),
    ("sexual_body_count", "Sex/body coded terms"),
    ("nigerian_pidgin_count", "Nigerian/Pidgin markers"),
    ("religious_spiritual_count", "Religious/spiritual markers"),
    ("politics_economy_count", "Politics/economy markers"),
    ("youth_culture_count", "Youth culture markers"),
    ("special_char_density", "Special character density"),
    ("unique_word_ratio", "Lexical diversity"),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for ax, (feature, title) in zip(axes, plot_features):
    data = [feat_df[feat_df["label"] == label][feature].values for label in LABEL_ORDER]
    parts = ax.violinplot(data, showmedians=True, showextrema=False)
    for body, label in zip(parts["bodies"], LABEL_ORDER):
        body.set_facecolor(PALETTE[label])
        body.set_alpha(0.75)
    parts["cmedians"].set_color("black")
    ax.set_xticks(range(1, len(LABEL_ORDER) + 1))
    ax.set_xticklabels([x.replace("_", "\n") for x in LABEL_ORDER], fontsize=6)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

#plt.suptitle("Figure 5. Sociolinguistic feature distributions across expanded categories", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig5_sociolinguistic_features.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FIGURE 6: t-SNE EMBEDDING STRUCTURE
# =============================================================================

print("Computing t-SNE embedding...")
tfidf_full = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_vec = tfidf_full.fit_transform(df["text"].values).toarray()

pca = PCA(n_components=min(50, X_vec.shape[1] - 1), random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_vec)

tsne = TSNE(
    n_components=2,
    perplexity=40,
    max_iter=1000,
    random_state=RANDOM_STATE,
    learning_rate="auto",
    init="pca",
)
X_tsne = tsne.fit_transform(X_pca)

fig, ax = plt.subplots(figsize=(11, 8))
for label in LABEL_ORDER:
    mask = df["label"] == label
    ax.scatter(
        X_tsne[mask, 0],
        X_tsne[mask, 1],
        s=10,
        alpha=0.5,
        color=PALETTE[label],
        label=label.replace("_", " "),
    )

ax.set_xticks([])
ax.set_yticks([])
#ax.set_title("Figure 6. t-SNE visualization of expanded coded-discourse categories", fontweight="bold")
ax.legend(fontsize=8, markerscale=2, loc="best")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig6_tsne_expanded_categories.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FIGURE 7: LDA TOPIC MODELING
# =============================================================================

print("Running LDA topic modeling...")
count_vec = CountVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
)
X_counts = count_vec.fit_transform(df["text"].values)
count_words = count_vec.get_feature_names_out()

n_topics = 8
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=RANDOM_STATE,
    max_iter=25,
    learning_method="online",
    n_jobs=-1,
)
lda.fit(X_counts)

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for topic_idx, ax in enumerate(axes):
    top_idx = lda.components_[topic_idx].argsort()[-12:][::-1]
    words = [count_words[i] for i in top_idx]
    weights = lda.components_[topic_idx][top_idx]
    weights = weights / weights.max()

    ax.barh(range(len(words)), weights[::-1], color="#457B9D")
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words[::-1], fontsize=8)
    ax.set_title(f"Topic {topic_idx + 1}", fontweight="bold")
    ax.set_xlabel("Relative weight")
    ax.spines[["top", "right"]].set_visible(False)

#plt.suptitle("Figure 7. LDA topic modeling of expanded algospeak/coded discourse corpus", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig7_lda_topics.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FIGURE 8: FEATURE CORRELATION
# =============================================================================

corr_cols = [
    "substitution_count",
    "special_char_density",
    "global_algospeak_count",
    "sexual_body_count",
    "nigerian_pidgin_count",
    "religious_spiritual_count",
    "politics_economy_count",
    "youth_culture_count",
    "unique_word_ratio",
    "avg_word_len",
]

corr = feat_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in corr_cols], fontsize=7)
ax.set_yticklabels([c.replace("_", " ") for c in corr_cols], fontsize=8)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

#ax.set_title("Figure 8. Correlation among sociolinguistic marker features", fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(FIGDIR, "fig8_feature_correlation.png"), dpi=DPI, bbox_inches="tight")
plt.close()


# =============================================================================
# FINAL PRINT SUMMARY
# =============================================================================

print("\nAll outputs saved to:", FIGDIR)
print("\n=== FINAL MODEL PERFORMANCE SUMMARY ===")
print(summary_df.round(3))

print("\nFiles written:")
for fname in sorted(os.listdir(FIGDIR)):
    print(" -", fname)


Corpus size: 3600 texts
label
nigerian_pidgin        450
religious_spiritual    450
ambiguous              450
global_algospeak       450
youth_culture          450
standard               450
politics_economy       450
sexual_body            450
Name: count, dtype: int64
Feature matrix: (3600, 24)

Training models...
Naive Bayes            | CV F1=0.970±0.007 | Test Acc=0.975 | Weighted F1=0.975
Logistic Regression    | CV F1=0.968±0.007 | Test Acc=0.975 | Weighted F1=0.975
Linear SVM             | CV F1=0.978±0.006 | Test Acc=0.983 | Weighted F1=0.983
Random Forest          | CV F1=0.881±0.010 | Test Acc=0.906 | Weighted F1=0.906
Gradient Boosting      | CV F1=0.961±0.008 | Test Acc=0.971 | Weighted F1=0.971

Best model: Linear SVM
Computing t-SNE embedding...
Running LDA topic modeling...

All outputs saved to: figures_expanded_algospeak

=== FINAL MODEL PERFORMANCE SUMMARY ===
                     CV F1 Mean  CV F1 SD  Test Accuracy  Weighted Precision  \
Naive Bayes               0